In [1]:
deposit = 200000
interest_rate = 4.85
P = 475000 # loan principal
N = 12*30 # loan term
r = (interest_rate / 100) / 12
c = (P * r) / (1 - (1 + r)**-N) # monthly repayment

C = 4552.99 + 28001.45 # rates + body corp fees

def principal_remaining(P, r, c, n):
    return P * (1 + r)**n - c * (((1 + r)**n - 1) / r)
def interest_paid(P, r, c, n):
    return (c * n) - (P - principal_remaining(P, r, c, n))
def total_cost(deposit, C, c, N):
    return deposit + (C * (N / 12)) + (c * N)



# Rent vs Mortgage Comparison

Compare renting at $600/week vs buying with a mortgage.

**Key Formula:**  
You're better off buying if:  
`Rent Paid > (Interest Paid + Ownership Costs - Principal Paid Off)`

Or equivalently, your **net cost** of ownership is less than rent.

In [2]:
# Rent parameters
weekly_rent = 600
annual_rent = weekly_rent * 52
monthly_rent = annual_rent / 12

print(f"Weekly rent: ${weekly_rent:,.2f}")
print(f"Monthly rent: ${monthly_rent:,.2f}")
print(f"Annual rent: ${annual_rent:,.2f}")

Weekly rent: $600.00
Monthly rent: $2,600.00
Annual rent: $31,200.00


In [3]:
# Comparison function for any time period (in months)
def compare_rent_vs_mortgage(n_months):
    """
    Compare renting vs mortgage over n_months period.
    
    Returns:
    - Total rent paid
    - Total interest paid
    - Total ownership costs (rates + body corp)
    - Principal paid off
    - Net cost of ownership (interest + costs - principal)
    - Savings (positive means mortgage is better)
    """
    # Rent
    total_rent = monthly_rent * n_months
    
    # Mortgage
    total_interest = interest_paid(P, r, c, n_months)
    total_ownership_costs = C * (n_months / 12)
    principal_paid = P - principal_remaining(P, r, c, n_months)
    
    # Net cost of owning vs renting
    net_ownership_cost = total_interest + total_ownership_costs - principal_paid
    savings = total_rent - net_ownership_cost
    
    return {
        'months': n_months,
        'years': n_months / 12,
        'total_rent': total_rent,
        'total_interest': total_interest,
        'total_ownership_costs': total_ownership_costs,
        'principal_paid': principal_paid,
        'net_ownership_cost': net_ownership_cost,
        'savings': savings,
        'better_option': 'Mortgage' if savings > 0 else 'Rent'
    }

## Example Comparisons at Different Time Periods

In [4]:
# Compare at different time periods
import pandas as pd

time_periods = [12, 24, 60, 120, 180, 240, 360]  # months (1, 2, 5, 10, 15, 20, 30 years)
results = [compare_rent_vs_mortgage(months) for months in time_periods]

df = pd.DataFrame(results)
df['savings'] = df['savings'].round(2)
df['net_ownership_cost'] = df['net_ownership_cost'].round(2)
df['total_rent'] = df['total_rent'].round(2)

# Format for display
pd.set_option('display.float_format', lambda x: f'${x:,.2f}' if abs(x) > 0.01 else '$0.00')
print(df[['years', 'total_rent', 'net_ownership_cost', 'savings', 'better_option']].to_string(index=False))
pd.reset_option('display.float_format')

 years  total_rent  net_ownership_cost      savings better_option
 $1.00  $31,200.00          $48,233.72  $-17,033.72          Rent
 $2.00  $62,400.00          $95,753.35  $-33,353.35          Rent
 $5.00 $156,000.00         $233,664.66  $-77,664.66          Rent
$10.00 $312,000.00         $445,561.63 $-133,561.63          Rent
$15.00 $468,000.00         $629,730.71 $-161,730.71          Rent
$20.00 $624,000.00         $778,579.78 $-154,579.78          Rent
$30.00 $936,000.00         $928,986.23    $7,013.77      Mortgage


## Detailed Breakdown for a Specific Period

In [6]:
# Detailed analysis for 5 years
analysis = compare_rent_vs_mortgage(60)

print(f"=== RENT VS MORTGAGE COMPARISON ({analysis['years']:.0f} years) ===\n")
print(f"RENTING:")
print(f"  Total rent paid: ${analysis['total_rent']:,.2f}\n")

print(f"BUYING:")
print(f"  Total interest paid: ${analysis['total_interest']:,.2f}")
print(f"  Total ownership costs: ${analysis['total_ownership_costs']:,.2f}")
print(f"  Principal paid off: ${analysis['principal_paid']:,.2f}")
print(f"  ─────────────────────────────")
print(f"  Net ownership cost: ${analysis['net_ownership_cost']:,.2f}\n")

print(f"COMPARISON:")
if analysis['savings'] > 0:
    print(f"  ✓ Mortgage is better by ${analysis['savings']:,.2f}")
else:
    print(f"  ✗ Rent is better by ${-analysis['savings']:,.2f}")
    
print(f"\nBreakdown:")
print(f"  Rent paid: ${analysis['total_rent']:,.2f}")
print(f"  Minus net ownership cost: -${analysis['net_ownership_cost']:,.2f}")
print(f"  = Your savings: ${analysis['savings']:,.2f}")

=== RENT VS MORTGAGE COMPARISON (5 years) ===

RENTING:
  Total rent paid: $156,000.00

BUYING:
  Total interest paid: $110,642.32
  Total ownership costs: $162,772.20
  Principal paid off: $39,749.85
  ─────────────────────────────
  Net ownership cost: $233,664.66

COMPARISON:
  ✗ Rent is better by $77,664.66

Breakdown:
  Rent paid: $156,000.00
  Minus net ownership cost: -$233,664.66
  = Your savings: $-77,664.66


## Considering Opportunity Cost of Deposit

If you rent instead of buying, your $200k deposit could be invested elsewhere (e.g., index funds, high-interest savings).

**Alternative comparison:**
- **Scenario A (Rent):** Pay rent + Invest deposit elsewhere
- **Scenario B (Buy):** Pay mortgage + Build equity

The question becomes: Is `(Equity Built - Deposit Invested Growth)` worth the difference in ongoing costs?

In [7]:
# Enhanced comparison including opportunity cost of deposit
def compare_with_opportunity_cost(n_months, investment_return_rate=0.05):
    """
    Compare rent vs mortgage including opportunity cost of the deposit.
    
    investment_return_rate: annual return rate if deposit was invested (e.g., 0.05 = 5%)
    """
    basic = compare_rent_vs_mortgage(n_months)
    
    # If renting, the deposit could be invested
    years = n_months / 12
    invested_deposit_value = deposit * (1 + investment_return_rate) ** years
    investment_gains = invested_deposit_value - deposit
    
    # Adjusted comparison
    # Renting: Pay rent, but gain investment returns on deposit
    rent_scenario_net_cost = basic['total_rent'] - investment_gains
    
    # Buying: Net ownership cost, but you've used your deposit
    buy_scenario_net_cost = basic['net_ownership_cost']
    
    adjusted_savings = rent_scenario_net_cost - buy_scenario_net_cost
    
    return {
        **basic,
        'investment_return_rate': investment_return_rate * 100,
        'invested_deposit_value': invested_deposit_value,
        'investment_gains': investment_gains,
        'rent_scenario_net_cost': rent_scenario_net_cost,
        'buy_scenario_net_cost': buy_scenario_net_cost,
        'adjusted_savings': adjusted_savings,
        'adjusted_better_option': 'Mortgage' if adjusted_savings > 0 else 'Rent'
    }

# Example: 5 years with 5% annual return on invested deposit
analysis_adj = compare_with_opportunity_cost(60, 0.05)

print(f"=== ADJUSTED COMPARISON (5 years, 5% investment return) ===\n")
print(f"SCENARIO A: RENT + INVEST DEPOSIT")
print(f"  Total rent paid: ${analysis_adj['total_rent']:,.2f}")
print(f"  Deposit invested grows to: ${analysis_adj['invested_deposit_value']:,.2f}")
print(f"  Investment gains: ${analysis_adj['investment_gains']:,.2f}")
print(f"  Net cost: ${analysis_adj['rent_scenario_net_cost']:,.2f}\n")

print(f"SCENARIO B: BUY WITH MORTGAGE")
print(f"  Interest + Costs - Principal: ${analysis_adj['buy_scenario_net_cost']:,.2f}\n")

print(f"RESULT:")
if analysis_adj['adjusted_savings'] > 0:
    print(f"  ✓ Mortgage is still better by ${analysis_adj['adjusted_savings']:,.2f}")
else:
    print(f"  ✗ Renting + investing is better by ${-analysis_adj['adjusted_savings']:,.2f}")
    
print(f"\nNote: This assumes you could earn {analysis_adj['investment_return_rate']:.1f}% annual return on your ${deposit:,.0f} deposit.")

=== ADJUSTED COMPARISON (5 years, 5% investment return) ===

SCENARIO A: RENT + INVEST DEPOSIT
  Total rent paid: $156,000.00
  Deposit invested grows to: $255,256.31
  Investment gains: $55,256.31
  Net cost: $100,743.69

SCENARIO B: BUY WITH MORTGAGE
  Interest + Costs - Principal: $233,664.66

RESULT:
  ✗ Renting + investing is better by $132,920.98

Note: This assumes you could earn 5.0% annual return on your $200,000 deposit.


## The Leverage Effect: Property Capital Gains

**This is the key advantage of buying!** 

With a mortgage, you control a $675k property with only $200k down (3.4x leverage).

- If property grows 3%/year, you gain 3% of $675k = ~$20k/year
- If deposit invested grows 5%/year, you gain 5% of $200k = $10k/year

Even though 5% > 3%, the leverage means property gains are much larger in absolute terms!

Since you're assuming property prices and rent move in parallel, we should include property appreciation.

In [8]:
# Complete comparison with leverage and capital gains
def complete_comparison(n_months, property_growth_rate=0.03, investment_return_rate=0.05):
    """
    Full comparison including:
    - Rent vs ownership costs
    - Opportunity cost of deposit
    - Property capital gains (with leverage!)
    - Investment returns on deposit
    
    property_growth_rate: annual property appreciation rate
    investment_return_rate: annual return if deposit invested elsewhere
    """
    basic = compare_rent_vs_mortgage(n_months)
    years = n_months / 12
    
    # Total property value (purchase price)
    property_value = deposit + P
    
    # SCENARIO A: RENT + INVEST DEPOSIT
    invested_deposit_value = deposit * (1 + investment_return_rate) ** years
    investment_gains = invested_deposit_value - deposit
    rent_scenario_wealth = invested_deposit_value  # Your wealth from investing
    rent_scenario_cost = basic['total_rent']
    
    # SCENARIO B: BUY WITH MORTGAGE
    property_future_value = property_value * (1 + property_growth_rate) ** years
    property_capital_gains = property_future_value - property_value
    remaining_loan = principal_remaining(P, r, c, n_months)
    equity = property_future_value - remaining_loan  # Property value minus what you owe
    
    buy_scenario_wealth = equity  # Your wealth in the property
    buy_scenario_cost = basic['total_interest'] + basic['total_ownership_costs']
    
    # NET COMPARISON
    # Scenario A: You have invested deposit, paid rent
    scenario_a_net = rent_scenario_wealth - rent_scenario_cost
    
    # Scenario B: You have equity in property, paid interest + costs
    scenario_b_net = buy_scenario_wealth - buy_scenario_cost
    
    wealth_difference = scenario_b_net - scenario_a_net
    
    return {
        'years': years,
        'property_growth_rate': property_growth_rate * 100,
        'investment_return_rate': investment_return_rate * 100,
        # Rent scenario
        'total_rent': rent_scenario_cost,
        'invested_deposit_value': invested_deposit_value,
        'investment_gains': investment_gains,
        'scenario_a_net_wealth': scenario_a_net,
        # Buy scenario
        'total_interest': basic['total_interest'],
        'total_ownership_costs': basic['total_ownership_costs'],
        'property_future_value': property_future_value,
        'property_capital_gains': property_capital_gains,
        'remaining_loan': remaining_loan,
        'equity': equity,
        'scenario_b_net_wealth': scenario_b_net,
        # Comparison
        'wealth_difference': wealth_difference,
        'better_option': 'Mortgage' if wealth_difference > 0 else 'Rent'
    }

# Example with realistic rates
result = complete_comparison(60, property_growth_rate=0.03, investment_return_rate=0.05)

print(f"=== COMPLETE COMPARISON ({result['years']:.0f} years) ===")
print(f"Property growth: {result['property_growth_rate']:.1f}% | Investment return: {result['investment_return_rate']:.1f}%\n")

print(f"SCENARIO A: RENT + INVEST ${deposit:,.0f}")
print(f"  Total rent paid: ${result['total_rent']:,.2f}")
print(f"  Deposit grows to: ${result['invested_deposit_value']:,.2f}")
print(f"  Investment gains: ${result['investment_gains']:,.2f}")
print(f"  ─────────────────────────────")
print(f"  Net wealth position: ${result['scenario_a_net_wealth']:,.2f}\n")

print(f"SCENARIO B: BUY WITH MORTGAGE")
print(f"  Interest paid: ${result['total_interest']:,.2f}")
print(f"  Ownership costs: ${result['total_ownership_costs']:,.2f}")
print(f"  Property value: ${result['property_future_value']:,.2f}")
print(f"  Capital gains: ${result['property_capital_gains']:,.2f}")
print(f"  Remaining loan: ${result['remaining_loan']:,.2f}")
print(f"  Your equity: ${result['equity']:,.2f}")
print(f"  ─────────────────────────────")
print(f"  Net wealth position: ${result['scenario_b_net_wealth']:,.2f}\n")

print(f"RESULT:")
if result['wealth_difference'] > 0:
    print(f"  ✓ BUYING is better by ${result['wealth_difference']:,.2f}")
    print(f"    You end up ${result['wealth_difference']:,.2f} wealthier by buying!")
else:
    print(f"  ✗ RENTING is better by ${-result['wealth_difference']:,.2f}")
    print(f"    You end up ${-result['wealth_difference']:,.2f} wealthier by renting!")
    
print(f"\n💡 The leverage effect: Your ${deposit:,.0f} deposit controls a ${deposit + P:,.0f} property!")
print(f"   Property gains: ${result['property_capital_gains']:,.2f} vs Investment gains: ${result['investment_gains']:,.2f}")

=== COMPLETE COMPARISON (5 years) ===
Property growth: 3.0% | Investment return: 5.0%

SCENARIO A: RENT + INVEST $200,000
  Total rent paid: $156,000.00
  Deposit grows to: $255,256.31
  Investment gains: $55,256.31
  ─────────────────────────────
  Net wealth position: $99,256.31

SCENARIO B: BUY WITH MORTGAGE
  Interest paid: $110,642.32
  Ownership costs: $162,772.20
  Property value: $782,510.00
  Capital gains: $107,510.00
  Remaining loan: $435,250.15
  Your equity: $347,259.85
  ─────────────────────────────
  Net wealth position: $73,845.34

RESULT:
  ✗ RENTING is better by $25,410.98
    You end up $25,410.98 wealthier by renting!

💡 The leverage effect: Your $200,000 deposit controls a $675,000 property!
   Property gains: $107,510.00 vs Investment gains: $55,256.31


## Sensitivity Analysis: Testing Different Growth Rates

In [9]:
# Test different property growth rates
print("=== HOW MUCH PROPERTY GROWTH DO YOU NEED TO BREAK EVEN? (5 years) ===\n")
print("Property Growth | Investment Return | Wealth Difference | Winner")
print("─" * 70)

for prop_rate in [0.01, 0.02, 0.03, 0.04, 0.05]:
    result = complete_comparison(60, property_growth_rate=prop_rate, investment_return_rate=0.05)
    winner = "🏠 Mortgage" if result['wealth_difference'] > 0 else "🏢 Rent+Invest"
    print(f"    {prop_rate*100:>4.1f}%       |       5.0%         | ${result['wealth_difference']:>13,.2f} | {winner}")

print("\n💡 Key insight: Due to leverage, even modest property growth")
print("   (2-3%/year) can outperform higher investment returns (5%/year)!")

=== HOW MUCH PROPERTY GROWTH DO YOU NEED TO BREAK EVEN? (5 years) ===

Property Growth | Investment Return | Wealth Difference | Winner
──────────────────────────────────────────────────────────────────────
     1.0%       |       5.0%         | $   -98,489.19 | 🏢 Rent+Invest
     2.0%       |       5.0%         | $   -62,666.43 | 🏢 Rent+Invest
     3.0%       |       5.0%         | $   -25,410.98 | 🏢 Rent+Invest
     4.0%       |       5.0%         | $    13,319.73 | 🏠 Mortgage
     5.0%       |       5.0%         | $    53,569.08 | 🏠 Mortgage

💡 Key insight: Due to leverage, even modest property growth
   (2-3%/year) can outperform higher investment returns (5%/year)!
